In [ ]:
from quspin.basis import spin_basis_1d # 用于创建一维自旋-1/2链的希尔伯特空间基；
from quspin.basis import spinful_fermion_basis_1d # 用于创建一维1/2自旋费米子链的希尔伯特空间基；
from quspin.operators import hamiltonian # 用于在给定的基（basis）上构建哈密顿量算符(或其他物理观测量)；
from Physical_Model_on_Honeycomb import tJ_Model_honeycomb_16site
from Physical_Model_on_Honeycomb import Heisenberg_Model_honeycomb
from Physical_Model_on_Honeycomb import tJ_Model_honeycomb_8site
from Physical_Model_on_Honeycomb import tJ_Model_honeycomb_Correlation_function_16site
import numpy as np 
import matplotlib.pyplot as plt  # 用于结果可视化
from matplotlib.ticker import MultipleLocator # MultipleLocator是matplotlib中的一个刻度定位器类,用来按照指定的倍数来设置坐标轴刻度
import ast # 导入Python的ast（抽象语法树）模块，用于安全地解析字符串形式的Python数据结构

In [ ]:
#----------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------########### t-J Model on honeycomb(16格点) ################----------------------------------------
print('='*80)
print('t-J Model的严格对角化')
print()

#### honeycomb晶格参数
L=16 # 格点数
bond_number = 22 # honeycomb晶格的两格点之间bond数

# 经常修改的参数(包括海森堡模型参数)
hole_doping = 4/16 # 空穴率，即空穴数与格点数的比值；
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.2 # 相互作用系数；

#### 其他参数设置
t2 = 0.0 # 次近邻跃迁项(hopping)系数(默认为0)；
J2 = 0.0 # 次近邻相互作用系数(默认为0)；
kblock=None # 动量块即波矢k的取值(平移对称性的量子数，对应动量2πk/L)，取int(默认为None)；
pblock=None # 空间反射(宇称)对称性的量子数，1对应偶宇称、-1对应奇宇称(默认为None)；
sblock=None # 自旋反演对称性的量子数,1对应偶对称性、-1对应对称性(默认为None)；
a=1 # 原胞大小(即原胞格点数)。当 a>1 时，系统被视为具有 a-site 的晶胞结构(默认为1)；
dtype=np.complex128 # 设置矩阵的数据类型(默认np.complex128，即128位复数)

#### 哈密顿量的构建
H, basis = tJ_Model_honeycomb_16site(hole_doping, t1, J1, t2, J2, kblock, pblock, sblock, a, dtype)

## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状

####### 具体计算
#### 1. 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)

#### 2. 自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓
    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    Sz_i = S_z_i.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED.append(Sz_i)


#### 3. 密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    n_i = ni.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    n_ED.append(n_i)

#### 将以上测量值打印
print("\n============= 测量结果 ==============")
print(f"\nt-J Model基态能量: {E_gs:.15f}")
print("\n-------- 格点期望值 ---------")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i+1:<10} {Sz_ED[i]:<25.15f} {n_ED[i]:<20.15f}")

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------########### t-J Model on honeycomb(8格点) ################----------------------------------------
# 参数设置
L = 8 # 格点数
hole_doping = 2/8 # 空穴率
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.2 # 最近邻相互作用系数；

t2 = 0.0 # 次近邻跃迁项(hopping)系数；
J2 = 0.0 # 次近邻相互作用系数；

#### 哈密顿量的构建
H, basis = tJ_Model_honeycomb_8site(hole_doping, t1, J1, t2, J2)

print('='*80)
print('t-J Model的严格对角化')
print()

## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状s

####### 具体计算
#### 1. 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)

#### 2. 自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓
    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值
    Sz_i = S_z_i.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED.append(Sz_i)


#### 3. 密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED = []  # 创建存储每个格点的密度n平均值

for i in range(L):
    # 构建第i个格点的n算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值
    n_i = ni.expt_value(V_gs).real
    n_ED.append(n_i)

#### 将以上测量值打印
print("\n============= 测量结果 ==============")
print(f"\nt-J Model基态能量: {E_gs:.15f}")
print("\n-------- 格点期望值 ---------")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i+1:<10} {Sz_ED[i]:<25.15f} {n_ED[i]:<20.15f}")

In [ ]:
### DMRG计算结果
# 输入数据
data_n = """
[
  [[0], [0.892861409245,              0]],
  [[1], [0.950667632455,              0]],
  [[2], [ 0.89286140924,              0]],
  [[3], [0.950667632446,              0]],
  [[4], [0.820003851273,              0]],
  [[5], [0.836467107035,              0]],
  [[6], [0.820003851273,              0]],
  [[7], [0.836467107033,              0]],
  [[8], [0.820003851275,              0]],
  [[9], [0.836467107032,              0]],
  [[10], [0.820003851277,              0]],
  [[11], [0.836467107033,              0]],
  [[12], [ 0.89286140924,              0]],
  [[13], [0.950667632453,              0]],
  [[14], [ 0.89286140924,              0]],
  [[15], [0.950667632451,              0]]
]
"""
data_Sz = """
[
  [[0], [-1.15151353924e-07,              0]],
  [[1], [-4.48442671885e-12,              0]],
  [[2], [1.15152933822e-07,              0]],
  [[3], [2.50034612981e-12,              0]],
  [[4], [1.96946194256e-07,              0]],
  [[5], [-2.89088457839e-14,              0]],
  [[6], [-1.96946181389e-07,              0]],
  [[7], [4.16128503183e-13,              0]],
  [[8], [-2.81862097872e-07,              0]],
  [[9], [-9.89017895359e-14,              0]],
  [[10], [2.81861054915e-07,              0]],
  [[11], [2.88817323464e-13,              0]],
  [[12], [1.6471069509e-07,              0]],
  [[13], [-2.29967520721e-12,              0]],
  [[14], [-1.64705964645e-07,              0]],
  [[15], [-1.57399093759e-12,              0]]
]
"""

# 提取出Sz值并储存
parsed_data_Sz = ast.literal_eval(data_Sz) # 通过ast.literal_eval安全将字符串data转换为实际的Python列表对象
Sz_DMRG = [item[1][0] for item in parsed_data_Sz] # 使用列表推导式从解析后的数据中提取特定值

# 提取出密度n值并储存
parsed_data_n = ast.literal_eval(data_n) 
n_DMRG = [item[1][0] for item in parsed_data_n] 

# 计算ED与DMRG之间的误差
Sz_relative_error = [] # 创建存储每个格点的S_z相对误差
n_relative_error = [] # 创建存储每个格点的n相对误差
Sz_absolute_error = [] # 创建存储每个格点的S_z绝对误差
n_absolute_error = [] # 创建存储每个格点的n绝对误差
for i in range(L):
    Sz_re_err = abs((Sz_ED[i] - Sz_DMRG[i])/Sz_ED[i])
    n_re_err = abs((n_ED[i] - n_DMRG[i])/n_ED[i])
    Sz_relative_error.append(Sz_re_err)
    n_relative_error.append(n_re_err)
    
    Sz_abs_err = abs(Sz_ED[i] - Sz_DMRG[i])
    n_abs_err = abs(n_ED[i] - n_DMRG[i])
    Sz_absolute_error.append(Sz_abs_err)
    n_absolute_error.append(n_abs_err)

# 总相对误差(即每个格点相对误差的和)
Sz_total_relative_error = sum(Sz_relative_error)
n_total_relative_error = sum(n_relative_error)
print(f'\n自旋总相对误差: {Sz_total_relative_error:.5e}')
print(f'密度总相对误差: {n_total_relative_error:.5e}')

# 总相对误差(即每个格点相对误差的和)
Sz_total_absolute_error = sum(Sz_absolute_error)
n_total_absolute_error = sum(n_absolute_error)
print(f'\n自旋总绝对误差: {Sz_total_absolute_error:.5e}')
print(f'密度总绝对误差: {n_total_absolute_error:.5e}')


#---------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 自旋Sz算符平均值 ################---------------------------------------
print()
print('='*80)
print('每个格点的自旋平均值 <S^z_i>：')
print()

## 可视化S_z平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), Sz_ED, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), Sz_DMRG, 'ys-', linewidth=2, markersize=8, label="DMRG")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle S_i^z \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average spin per site $\\langle S_i^z \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

## 可视化S_z相对误差
plt.figure(figsize=(10, 5))
plt.plot(range(L), Sz_absolute_error, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('Sz absolute error', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('The absolute error of average spin per site', fontsize=20)
plt.grid(True, alpha=0.3)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()



#--------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 密度算符平均值 ################----------------------------------------
print()
print('='*80)
print('每个格点的密度平均值 <n_i>：')
print()

## 可视化n平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), n_ED, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), n_DMRG, 'ys-', linewidth=2, markersize=8, label="DMRG")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle n_i \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average density per site $\\langle n_i \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

## 可视化密度n相对误差
plt.figure(figsize=(10, 5))
plt.plot(range(L), n_absolute_error, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('n absolute error', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('The absolute error of average density per site', fontsize=20)
plt.grid(True, alpha=0.3)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------###########  Heisenberg Model on honeycomb(16格点) ################----------------------------------------
print('='*80)
print('Heisenberg Model的严格对角化')
print()

#### Heisenberg Model特有参数设置
Nup=None # 系统自旋向上的总数；
zblock=None # 自旋反演对称性的量子数,1对应偶对称性、-1对应对称性(默认为None)；
a=1 # 原胞大小(即原胞格点数)。当 a>1 时，系统被视为具有 a-site 的晶胞结构(默认为1)；
dtype=np.float64 # 设置矩阵的数据类型(默认np.float64)

#### 哈密顿量的构建
H_Heisenberg, basis_Heisenberg = Heisenberg_Model_honeycomb(J1, J2, Nup, kblock, pblock, zblock, a, dtype)

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H_Heisenberg))
print("希尔伯特空间维度:", H_Heisenberg.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H_Heisenberg.shape) # H作为运算符的矩阵形状s

## 基态能量
E_gs_Heisenberg, V_gs_Heisenberg = H_Heisenberg.eigsh(k=1, which='SA') 
E_gs_Heisenberg = E_gs_Heisenberg[0] 
V_gs_Heisenberg = V_gs_Heisenberg[:, 0] 
print('-'*80)
print('Heisenberg Model基态能量：')
print(f"基态能量: {E_gs_Heisenberg:.15f}")
print(f"每格点能量: {E_gs_Heisenberg/L:.15f}")

# t-J模型半满对应海森堡模型基态能量
E_gs_tJ = E_gs_Heisenberg - 1/4 * J1 * bond_number
print('-'*80)
print('t-J模型半满对应海森堡模型基态能量：')
print()
print(f"基态能量: {E_gs_tJ:.15f}")
print(f"每格点能量: {E_gs_tJ/L:.15f}")

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------
#---------------------------------########### Heisenberg Model on honeycomb(8格点) ################----------------------------------------
# 参数设置
L = 8 # 格点数
bond_number = 10 # honeycomb晶格的两格点之间bond数
J1 = 0.05 # 相互作用系数；

#### 构建基矢
basis_Heisenberg = spin_basis_1d(
    L, 
    S="1/2",
    pauli=False
)

### 构建honeycomb的最近邻格点指标i、j的列表
lattice_nn_list = []
# honeycomb竖直方向(即y方向)指标列表的构建
for k in range(0, 5, 4):
    list_y = [(i,(i+1)) for i in range(k, k+3)] # x指标为k时对应的y方向指标列表
    lattice_nn_list.extend(list_y)
    periodic = (k+3,k) # 注：y方向为周期边界
    lattice_nn_list.append(periodic)

# honeycomb水平方向(即x方向)指标列表的构建
list_x = [(0,4), (2,6)] # 注：x方向为开放边界
lattice_nn_list.extend(list_x)

#### 定义site-coupling lists
# 最近邻耦合列表
nn_bond_list_xy = [[J1/2, i, j] for i,j in lattice_nn_list] # 自旋xy对应的耦合列表(将Sx与Sy用S+与S-表示则所有矩阵都是实矩阵，但会多出现1/2)；
nn_bond_list_zz = [[J1, i, j] for i,j in lattice_nn_list] # 自旋z对应的耦合列表；

## 构建哈密顿量
# 静态部分(不随时间变化的算符)
static = [
    # 最近邻海森堡相互作用(注意："x"对应 Sx、"y"对应 Sy、"z"对应 Sz、"+"对应 S+、"-"对应 S-)
    ["+-", nn_bond_list_xy], 
    ["-+", nn_bond_list_xy], 
    ["zz", nn_bond_list_zz],
]
# 动态部分(随时间变化的算符)
dynamic = [] 

H_Heisenberg = hamiltonian(  
    static, 
    dynamic, 
    basis=basis_Heisenberg, 
    dtype=np.float64,     
    check_symm=True,  
    check_pcon=True,  
    check_herm=True   
) 

print('='*80)
print('Heisenberg Model的严格对角化')
print()

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H_Heisenberg))
print("希尔伯特空间维度:", H_Heisenberg.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H_Heisenberg.shape) # H作为运算符的矩阵形状s

## 基态能量
E_gs_Heisenberg, V_gs_Heisenberg = H_Heisenberg.eigsh(k=1, which='SA') 
E_gs_Heisenberg = E_gs_Heisenberg[0] 
V_gs_Heisenberg = V_gs_Heisenberg[:, 0] 
print('-'*80)
print('Heisenberg Model基态能量：')
print(f"基态能量: {E_gs_Heisenberg:.15f}")
print(f"每格点能量: {E_gs_Heisenberg/L:.15f}")

# t-J模型半满对应海森堡模型基态能量
E_gs_tJ = E_gs_Heisenberg - 1/4 * J1 * bond_number
print('-'*80)
print('t-J模型半满对应海森堡模型基态能量：')
print()
print(f"基态能量: {E_gs_tJ:.15f}")
print(f"每格点能量: {E_gs_tJ/L:.15f}")



In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------########### t-J Model on honeycomb(4格点) ################----------------------------------------
# 参数设置
L = 4 # 格点数
hole_doping = 1/8 # 空穴率
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.2 # 相互作用系数；

#### 计算粒子数（根据空穴浓度）
# N_total = int(L * (1 - hole_doping))     # 总电子数；
# N_up = int(np.ceil(N_total / 2))    # 上自旋电子数(np.ceil()：向上取整函数)；
# N_down = int(np.floor(N_total / 2))  # 下自旋电子数(np.floor()：向下取整函数)；

N_up = 1
N_down = 0

#### 构建基矢：禁止双占据（每个格点最多一个电子）
basis = spinful_fermion_basis_1d(
    L, 
    Nf=(N_up, N_down), 
    double_occupancy=False
)

### 构建honeycomb的最近邻格点指标i、j的列表
lattice_nn_list = [(1,0), (0,2), (2,3)]

#### 定义site-coupling lists
### 最近邻hopping项
hop_nn_left = [[-t1, i, j] for i,j in lattice_nn_list]  # 直接项：从右向左跃迁项(𝑐†_𝑖,𝜎·𝑐_j,𝜎)；
hop_nn_right = [[t1, i, j] for i,j in lattice_nn_list]  # 厄密共轭项：从左向右跃迁项(𝑐†_j,𝜎·𝑐_𝑖,𝜎= - 𝑐_𝑖,𝜎·c†_j,𝜎)；

### 最近邻相互作用项(自旋相互作用项 + 密度相互作用项)：S_i·S_j-1/4 n_i·n_j = 1/2(S^+_i·S^-_j + S^-_i·S^+_j) + S^z_i·S^z_j - 1/4 n_i·n_j
## J * 1/2(S^+_i·S^-_j + S^-_i·S^+_j)项
int_ss = [[J1/2, i, j, i, j] for i,j in lattice_nn_list]
## J * (S^z_i·S^z_j - 1/4 n_i·n_j) = -J * 1/2(n_i↑·n_j↓ + n_j↑·n_i↓)项
int_nn_ij = [[-J1/2, i, j] for i,j in lattice_nn_list] # n_i↑·n_i↓项；
int_nn_ji = [[-J1/2, j, i] for i,j in lattice_nn_list] # n_j↑·n_i↓项；

# 构建static list
static = [
    ### 最近邻hopping项
    # 上自旋
    ["+-|", hop_nn_left],   # 右向hopping；
    ["-+|", hop_nn_right],  # 左向hopping(厄米共轭)；
    # 下自旋
    ["|+-", hop_nn_left],   # 右向hopping；
    ["|-+", hop_nn_right],  # 左向hopping(厄米共轭)；

    ### 最近邻相互作用项
    ## J * 1/2(S^+_i·S^-_j + S^-_i·S^+_j)项
    ["+-|-+", int_ss],  # S^+_i·S^-_j = 𝑐†_𝑖↑·𝑐_j↑·𝑐_𝑖↓·𝑐†_j↓；
    ["-+|+-", int_ss],  # S^-_i·S^+_j = 𝑐_𝑖↑·𝑐†_j↑·𝑐†_𝑖↓·𝑐_j↓；
    ## J * (S^z_i·S^z_j - 1/4 n_i·n_j) = -J * 1/2(n_i↑·n_j↓ + n_j↑·n_i↓)项
    ["n|n", int_nn_ij],  # n_i↑·n_j↓；
    ["n|n", int_nn_ji]   # n_j↑·n_i↓；
]

dynamic = []  # 无时间依赖项

# 构建哈密顿量
H = hamiltonian(
    static, 
    dynamic, 
    basis=basis, 
    dtype=np.complex128, 
    check_symm=True, 
    check_pcon=True, 
    check_herm=True
)

print('='*80)
print('t-J Model的严格对角化')
print()

## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状s

####### 具体计算
#### 1. 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)
print('-'*80)
print('t-J Model基态能量：')
print()
print(f"基态能量: {E_gs:.15f}")
print(f"每格点能量: {E_gs/L:.15f}")


In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------
#---------------------------------########### Heisenberg Model on honeycomb(4格点) ################----------------------------------------
# 参数设置
L = 4 # 格点数
bond_number = 3 # honeycomb晶格的两格点之间bond数
J1 = 0.2 # 相互作用系数；

#### 构建基矢
basis_Heisenberg = spin_basis_1d(
    L, 
    S="1/2",
    pauli=False
)

### 构建honeycomb的最近邻格点指标i、j的列表
lattice_nn_list = [(1,0), (0,2), (2,3)]

#### 定义site-coupling lists
# 最近邻耦合列表
nn_bond_list_xy = [[J1/2, i, j] for i,j in lattice_nn_list] # 自旋xy对应的耦合列表(将Sx与Sy用S+与S-表示则所有矩阵都是实矩阵，但会多出现1/2)；
nn_bond_list_zz = [[J1, i, j] for i,j in lattice_nn_list] # 自旋z对应的耦合列表；

## 构建哈密顿量
# 静态部分(不随时间变化的算符)
static = [
    # 最近邻海森堡相互作用(注意："x"对应 Sx、"y"对应 Sy、"z"对应 Sz、"+"对应 S+、"-"对应 S-)
    ["+-", nn_bond_list_xy], 
    ["-+", nn_bond_list_xy], 
    ["zz", nn_bond_list_zz],
]
# 动态部分(随时间变化的算符)
dynamic = [] 

H_Heisenberg = hamiltonian(  
    static, 
    dynamic, 
    basis=basis_Heisenberg, 
    dtype=np.float64,     
    check_symm=True,  
    check_pcon=True,  
    check_herm=True   
) 

print('='*80)
print('Heisenberg Model的严格对角化')
print()

#### 具体计算
## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H_Heisenberg))
print("希尔伯特空间维度:", H_Heisenberg.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H_Heisenberg.shape) # H作为运算符的矩阵形状s

## 基态能量
E_gs_Heisenberg, V_gs_Heisenberg = H_Heisenberg.eigsh(k=1, which='SA') 
E_gs_Heisenberg = E_gs_Heisenberg[0] 
V_gs_Heisenberg = V_gs_Heisenberg[:, 0] 
print('-'*80)
print('Heisenberg Model基态能量：')
print(f"基态能量: {E_gs_Heisenberg:.15f}")
print(f"每格点能量: {E_gs_Heisenberg/L:.15f}")

# t-J模型半满对应海森堡模型基态能量
E_gs_tJ = E_gs_Heisenberg - 1/4 * J1 * bond_number
print('-'*80)
print('t-J模型半满对应海森堡模型基态能量：')
print()
print(f"基态能量: {E_gs_tJ:.15f}")
print(f"每格点能量: {E_gs_tJ/L:.15f}")

